# 00 - Validate configuration and Azure authentication

This notebook loads local configuration, checks the required values without displaying secrets, authenticates with Microsoft Entra ID, and creates a preview-enabled Foundry project client.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

env_file = Path('.env')
if not env_file.exists():
    raise FileNotFoundError('Copy .env.example to .env and fill in the required values.')

load_dotenv(env_file)

required_variables = (
    'FOUNDRY_PROJECT_ENDPOINT',
    'APPLICATIONINSIGHTS_CONNECTION_STRING',
    'AGENT_NAME',
    'OTEL_AGENT_ID',
)
missing = [name for name in required_variables if not os.getenv(name)]
if missing:
    raise ValueError(f'Missing required environment variables: {missing}')

agent_name = os.environ['AGENT_NAME']
if not agent_name.replace('-', '').replace('_', '').isalnum():
    raise ValueError('AGENT_NAME may contain only letters, numbers, hyphens, and underscores.')

print(f'Configuration loaded for agent: {agent_name}')
print(f"OpenTelemetry agent ID: {os.environ['OTEL_AGENT_ID']}")
print('Application Insights connection string loaded: yes')

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=os.environ['FOUNDRY_PROJECT_ENDPOINT'],
    credential=credential,
    allow_preview=True,
)

# This request validates the endpoint, identity, and project access.
external_agents = list(project.agents.list(kind='external'))
print(f'Connected successfully. Existing external agents: {len(external_agents)}')